# Facility calibration — 01 source extraction

Single source of truth for the facility source register: shunting and parking
charges at service facilities. Every row is written by this notebook;
`data/sources_register.csv` is a generated artifact and must never be
hand-edited.

Run this before `02_facility_calibration.ipynb`, which fails rather than citing
a source that does not resolve against the register.

In [ ]:
# Facility calibration — source extraction
#
# Same contract as the TAC and energy pricing calibrations: notebook is truth,
# CSV is output. Documents shared with those domains (network statements that
# carry a track charge, an energy charge and a facility tariff) are re-declared
# here rather than cross-read, so each domain register stands alone.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    """Notebook may run from calib/ or from the repo root; resolve either."""
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/facility/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

# Date the register was last reviewed end to end.
REGISTER_REVIEWED = "2026-08-17"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "price_basis_year",
    "currency",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

## Facility tariffs

The eleven network statements and price lists that publish a shunting or
stabling charge. Seventeen further countries have none read yet and take the
European default — those are registered at the end so the gap is visible.

In [ ]:
# --- Facility tariffs ---
_R = REGISTER_REVIEWED

register_rows += [
    (
        "AT-SNNB-2026",
        "at_snnb_2026",
        "Used",
        "x",
        "Schienennetz-Nutzungsbedingungen 2026",
        "ÖBB-Infrastruktur AG",
        2024,
        2026,
        "EUR",
        "network_statement",
        "https://infrastruktur.oebb.at/de/geschaeftspartner/schienennetz/snnb",
        _R,
        "Tab.44 shunting leader with locomotive operation, and Tab.49 item "
        "4.2.2 the Abstellkapazitäten rate. The monthly booking rate is the "
        "realistic mode for a daily rotation",
    ),
    (
        "BE-NS-2027",
        "be_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (version 30 June 2026)",
        "Infrabel",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://infrabel.be/en/networkstatement",
        _R,
        "Appendix F.2 sheet 3.1.1.1 unit cost per access to a service "
        "facility; sheet 3.1.2.2 charges occupancy only in yards declared "
        "congested, which is why Belgium carries no parking charge",
    ),
    (
        "BG-NRIC-2026",
        "bg_nric_2026",
        "Used",
        "x",
        "Charges and Prices, Annex 5.3.2 v.06",
        "NRIC (National Railway Infrastructure Company)",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.rail-infra.bg/en/353",
        _R,
        "§2.6.2 draw-out and marshalling track use, and §12 stabling at 0.20 "
        "EUR per metre per 24 h — one of only four explicitly length-based "
        "stabling regimes in Europe",
    ),
    (
        "DE-APS-2027",
        "de_aps_2027",
        "Used",
        "x",
        "Anlagenpreissystem 2027 — Entgelte für Serviceeinrichtungen",
        "DB InfraGO",
        2025,
        2027,
        "EUR",
        "facility_price_list",
        "https://www.dbinfrago.com/web/schienennetz/regelwerke-nutzungsbedingungen/preise",
        _R,
        "§2.1 Zugbildung I train formation and Abstellung I stabling, both per "
        "started use-hour and expressly length-independent. Also the Elektrant "
        "and pre-heating energy rates this domain uses as the European hotel-"
        "power proxy. Facilities with an Anlagendisponent escalate the stabling "
        "charge by a factor rising 1 to 10 over the first ten hours",
    ),
    (
        "DK-NS-2027",
        "dk_ns_2027",
        "Used",
        "-",
        "Network Statement 2027",
        "Banedanmark",
        2026,
        2027,
        "DKK",
        "network_statement",
        "https://www.bane.dk/en/Railway/Network-Statement",
        _R,
        "§3.5 states no infrastructure charges are levied for operations or "
        "parking on sidings — a documented zero on both terms, not a gap",
    ),
    (
        "ES-ADIF-2027",
        "es_adif_2027",
        "Used",
        "x",
        "Declaración sobre la Red 2027 (NS ADIF V1)",
        "Adif / Adif Alta Velocidad",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.adif.es/sobre-adif/declaracion-red",
        _R,
        "Ch.6 basic operations: shunting driving 148 EUR/h and overall shunting "
        "operations 200 EUR/h. Adif sells the operation as a service, which is "
        "what makes Spain a full-scope country",
    ),
    (
        "GR-OSE-2026",
        "gr_ose_2026",
        "Used",
        "x",
        "Network Statement 2026 (EN final)",
        "OSE",
        2026,
        2019,
        "EUR",
        "network_statement",
        "https://ose.gr/wp-content/uploads/2026/02/OSE_2026-ENG_Final.pdf",
        _R,
        "§6.4.1 per manoeuvre, stating explicitly that the charge covers the "
        "manoeuvring team and not the locomotive or driver; §6.4.2 defines "
        "stabling as exactly two manoeuvres, with no time or length term",
    ),
    (
        "HR-NS-2027",
        "hr_ns_2027",
        "Used",
        "x",
        "Izvješće o mreži / Network Statement 2027",
        "HŽ Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://eng.hzinfra.hr/?page_id=284",
        _R,
        "§7.3.4.4 note 9, duration-banded per-metre rates with the first 24 h "
        "free on a side track",
    ),
    (
        "HU-NS-2627",
        "hu_ns_2627",
        "Used",
        "x",
        "Network Statement 2026-2027, Annex 5.2-6",
        "VPE / MÁV",
        2026,
        2027,
        "HUF",
        "network_statement",
        "https://vpe.kti.hu/en/network-statement/network-statement-2026-2027/",
        _R,
        "24,480 HUF per person-hour plus 79,752 HUF per vehicle-hour: the only "
        "infrastructure manager in the calibration that prices the shunting "
        "locomotive itself, which is why the Hungarian figure is 25 times the "
        "German one for the same movement",
    ),
    (
        "IT-NS-2027",
        "it_ns_2027",
        "Used",
        "x",
        "Network Statement 2027",
        "RFI",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.rfi.it/en/railway-infrastructure-access-/Network-statement.html",
        _R,
        "§5.4.6.1 indirect cost per parking operation. The energy element of "
        "the same section belongs to the energy pricing domain",
    ),
    (
        "NO-NS-2027",
        "no_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (EN v1.1)",
        "Bane NOR",
        2026,
        2026,
        "NOK",
        "network_statement",
        "https://oppslagsverk.banenor.no/en/network-statement/",
        _R,
        "Tab.9 §7.3.5: 6 NOK per hour per commenced 100 m outside Alnabru, "
        "with the first 48 h free — which zeroes a 12 h layover",
    ),
    (
        "PL-PLK-A91",
        "pl_plk_a91",
        "Used",
        "x",
        "Network Statement 2026/2027 Annex 9.1 (SMK)",
        "PKP Polskie Linie Kolejowe",
        2026,
        2027,
        "PLN",
        "network_statement",
        "https://en.plk-sa.pl/for-customers-and-partners/the-rules-for-allocating-train-paths/network-statement-2026/2027",
        _R,
        "Shunting at 3.66 PLN/km under electric traction, applied to "
        "standardised distances tabulated per location in Appendix 2.8",
    ),
    (
        "PT-IP-2027",
        "pt_ip_2027",
        "Used",
        "x",
        "1st Addenda to Network Statement 2027",
        "Infraestruturas de Portugal",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://servicos.infraestruturasdeportugal.pt/sites/default/files/1st%20Addenda%20Network%20Statement%202027_0.pdf",
        _R,
        "§5.4.4 long-duration shunting above 30 minutes (short duration 9.75 "
        "EUR) and stabling at 0.0405 EUR/min beyond the first hour, with "
        "timetabled technical stops exempt",
    ),
    (
        "SI-NS-2027",
        "si_ns_2027",
        "Used",
        "x",
        "Program omrežja / Network Statement 2027",
        "SŽ-Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://infrastruktura.sz.si/en/partners/access-to-infrastructure-for-rus/network-statement/",
        _R,
        "§5.4.2 P22 per departure from and arrival at the station of origin or "
        "a yard; §5.4.3 P23 charges only unplanned RU-attributable storage, so "
        "planned stabling is a documented zero",
    ),
]
print(f"{len(register_rows)} facility tariff documents")

## Cross-check, method and conversion sources

The two operator-side cost models the market top-up is calibrated against, the
labour band behind the index, and the FX and escalation references.

In [ ]:
# --- Cross-check, method and conversion sources ---
register_rows += [
    (
        "NOX-MODEL",
        "nox_model",
        "Used",
        "x",
        "Night train business case model, 2030 scenario",
        "Nox Mobility (publisher TO_VERIFY)",
        2024,
        2030,
        "EUR",
        "operator_model",
        "nox_model.xlsx",
        _R,
        "Line 213 OPS Infrastructure Access at 307.55 EUR/trip is the only "
        "operator-side figure that separates shunting and stabling from "
        "cleaning and servicing, which is what makes it the anchor for the "
        "market top-up. Lines 212 Track & Station Access and 214 Servicing "
        "are the neighbours it is separated from",
    ),
    (
        "RAMBOLL-BMDV",
        "ramboll_bmdv",
        "Used",
        "x",
        "Nachtzugstudie — Wirtschaftlichkeit von Nachtzugverbindungen",
        "Ramboll for BMDV",
        2025,
        2025,
        "EUR",
        "study",
        "https://bmdv.bund.de/",
        _R,
        "Train-driver hour priced at 62-104 EUR/h across fifteen countries: a "
        "1.68x spread, and the evidence behind the three-tier labour index. "
        "Its Reinigung/Abstellung block bundles cleaning with stabling and so "
        "cannot be used as a facility figure directly",
    ),
    (
        "ECB-FX",
        "ecb_fx",
        "Used",
        "-",
        "ECB euro foreign exchange reference rates",
        "European Central Bank",
        2026,
        2026,
        "EUR",
        "fx_reference",
        "https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html",
        _R,
        "Pinned at the same snapshot as the TAC and energy calibrations, so all "
        "three infrastructure domains reach EUR on identical terms",
    ),
    (
        "ECB-PROJECTIONS",
        "ecb_projections",
        "Used",
        "-",
        "Eurosystem staff macroeconomic projections for the euro area",
        "European Central Bank",
        2025,
        None,
        "EUR",
        "macro_projection",
        "https://www.ecb.europa.eu/press/projections/html/index.en.html",
        _R,
        "The HICP path both escalation rates are judged against: parking at "
        "HICP, shunting half a point above it on the wage argument",
    ),
    (
        "EUROSTAT-LCI",
        "eurostat_lci",
        "Used",
        "-",
        "Labour cost index by NACE Rev. 2 activity (lci_lci_r2_a)",
        "Eurostat",
        2026,
        None,
        "EUR",
        "official_statistics",
        "https://ec.europa.eu/eurostat/databrowser/view/lci_lci_r2_a/default/table",
        _R,
        "Transportation and storage labour costs have run consistently above "
        "headline HICP across the last decade, which is the evidence for "
        "escalating a labour-dominated charge faster than a land-and-track one",
    ),
]
print(f"{len(register_rows)} rows after method sources")

## Documents still to check

Seventeen countries take the European default shunting tariff because no
service-facility price list has been read for them, and the same countries
mostly take the default stabling rate. They are registered as `Not used` so
the gap is addressable rather than invisible — but note the priority: in every
one of them the market top-up dominates the total, so a price list moves the
all-in figure by only a few per cent.

In [ ]:
# --- Documents still to check ---
# Each of these countries needs a service-facility price list. Worth doing
# only for countries a chosen route set actually touches: the IM tariff is at
# most a fifth of the all-in figure everywhere it has been read.
_TO_CHECK = [
    (
        "CH-NZV",
        "ch_nzv",
        "SR 742.122 Eisenbahn-Netzzugangsverordnung (NZV)",
        "Swiss Confederation (Fedlex)",
        2026,
        2026,
        "CHF",
        "regulation",
        "https://www.fedlex.admin.ch/eli/cc/1999/142/de#a21",
    ),
    (
        "CZ-NS-2027",
        "cz_ns_2027",
        "Network Statement 2027 (EN web version)",
        "Správa železnic",
        2026,
        2027,
        "CZK",
        "network_statement",
        "https://www.spravazeleznic.cz/web/en/network-statement-2027",
    ),
    (
        "EE-TTJA-2026",
        "ee_ttja_2026",
        "Raudteeinfrastruktuuri kasutustasu määrad",
        "Tarbijakaitse ja Tehnilise Järelevalve Amet (TTJA)",
        2026,
        2026,
        "EUR",
        "tariff_decision",
        "https://ttja.ee/ariklient/raudtee/kasutustasu-maarad",
    ),
    (
        "FI-NS-2027",
        "fi_ns_2027",
        "Verkkoselostus / Network Statement 2027",
        "Väylävirasto",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.doria.fi/handle/10024/195216",
    ),
    (
        "FR-DRR-2027-A54",
        "fr_drr_2027_a54",
        "DRR 2027 Appendix 5.4 — service facilities",
        "SNCF Réseau",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.sncf-reseau.com/en/drr/network-statement-national-rail-network-timetable-2027",
    ),
    (
        "IE-NS-2027",
        "ie_ns_2027",
        "Network Statement 2027",
        "Iarnród Éireann",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.irishrail.ie/en-ie/about-us/iarnrod-eireann-network-statement",
    ),
    (
        "LT-LTG-2627",
        "lt_ltg_2627",
        "Network Statement 2026-2027 and annexes v1",
        "LTG Infra",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://ltginfra.lt/en/railway-infrastructure/map/network-statements/",
    ),
    (
        "LU-NS-2027",
        "lu_ns_2027",
        "Document de référence du réseau 2027 (EN v1.0)",
        "ACF / CFL",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://acf.gouvernement.lu/en/sillon/Document-de-reference-du-reseau.html",
    ),
    (
        "LV-NS-2027",
        "lv_ns_2027",
        "Network Statement 2027",
        "LDz / LatRailNet",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.ldz.lv/en/network-statement-2027",
    ),
    (
        "NL-NS-2027",
        "nl_ns_2027",
        "Network Statement 2027 (version 1.1)",
        "ProRail",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.prorail.nl/samenwerken/vervoerders/network-statement",
    ),
    (
        "RO-CFR-A25",
        "ro_cfr_a25",
        "Network Statement Annex 25.a / 26.a",
        "CFR SA",
        2025,
        2024,
        "RON",
        "network_statement",
        "https://cfr.ro/download-drr-2026-network-statement/",
    ),
    (
        "SE-NS-2027",
        "se_ns_2027",
        "Network Statement 2027",
        "Trafikverket",
        2026,
        2027,
        "SEK",
        "network_statement",
        "https://bransch.trafikverket.se/en/startpage/operations/Operations-railway/Network-Statement/network-statement-2027/",
    ),
    (
        "SK-ZSR-A52B",
        "sk_zsr_a52b",
        "Network Statement 2027 Annex 5.2.B",
        "ŽSR",
        2026,
        2019,
        "EUR",
        "network_statement",
        "https://www.zsr.sk/en/railway-undertaking/infrastructure/network-statement/network-statement-2027/",
    ),
    (
        "UK-NR-CP7",
        "uk_nr_cp7",
        "CP7 Track Usage Price List",
        "Network Rail",
        2024,
        2024,
        "GBP",
        "tariff_list",
        "https://www.networkrail.co.uk/industry-and-commercial/information-for-operators/network-statement/",
    ),
]
register_rows += [
    (
        sid,
        short,
        "Not used",
        "x",
        title,
        publisher,
        pub,
        basis,
        currency,
        kind,
        url,
        _R,
        "Service-facility section not yet read — the country takes the European "
        "default shunting tariff and stabling rate",
    )
    for sid, short, title, publisher, pub, basis, currency, kind, url in _TO_CHECK
]
print(f"{len(register_rows)} rows in total")

## Write and validate

In [ ]:
# --- Write and validate ---


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both calib notebooks."""
    path = DATA_DIR / name
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {name}: {len(rows)} rows")


register_dicts = [dict(zip(REGISTER_COLUMNS, r)) for r in register_rows]

ids = [r["source_id"] for r in register_dicts]
assert len(ids) == len(set(ids)), (
    f"duplicate source_id: {sorted({i for i in ids if ids.count(i) > 1})}"
)
_blank = [r["source_id"] for r in register_dicts if not r["url_or_file"]]
assert not _blank, f"rows with no url_or_file: {_blank}"

write_data("sources_register.csv", REGISTER_COLUMNS, register_dicts)
print(
    f"register: {len(register_dicts)} sources, "
    f"{sum(1 for r in register_dicts if r['used'] == 'Used')} in use, "
    f"{sum(1 for r in register_dicts if r['downloaded'] == 'x')} documents on disk"
)